In [1]:
from cell_labelling_functions import *

In [2]:
######### INPUTS ###########
# List of WSI paths
# WSI_paths = [
#     r'\\169.254.138.20\Andre\data\Stardist\Healthy nPOD\nondiabetic'
#     ]
# # date = '12_27_2024' # date of the segmentation
# date = '1_1_2025' # date of the segmentation
#     ]
WSI_paths = [
    r'\\kittyserverdw\Andre_kit\data\BTC\H1'
    ]
# # date = '12_27_2024' # date of the segmentation
# date = '5_3_2025' # date of the segmentation
date = '8_28_2025' # date of the segmentation
saveMASKS =0 # save segmented cells masks
target_resolution=1 # mask resolution 1->10x
RGBfeatures=1 # do you want to extract rgb features from contours 1-yes,0-no
gj=1 # do the geojson countours of the segmented cells, 1 if yes, 0 if no
inverse=0

# Change this to the model you want to load
# model_name = "panin_healthy"  #"pdac" "panin" "panin_healthy"  "monkey"  "fallopian_tube"
model_name = "pdac"  #"pdac" "panin" "panin_healthy"  "monkey"  "fallopian_tube"

############################
for WSI_path in WSI_paths:
    # Define output paths
    out_pth = os.path.join(WSI_path, f'StarDist_{date}_{model_name}')
    out_pth_json = os.path.join(out_pth, 'json')
    out_pth_tif = os.path.join(out_pth, 'stardist_masks')
    output_pixres = os.path.join(WSI_path,'segmentation_analysis', 'pix_res_info')
    outpthpkl = os.path.join(out_pth, 'stardist_feature_df_pickles_classes')
    outpthmat=os.path.join(outpthpkl,'mat')
    out_pth_contours = os.path.join(out_pth_json, 'geojsons', '32_polys_20x')
    out_pth_analysis = os.path.join(out_pth, 'analysis')
    
    # Create directories if they don't exist
    for path in [out_pth, out_pth_json, out_pth_tif,output_pixres,outpthpkl,outpthmat,out_pth_contours,out_pth_analysis]:
        os.makedirs(path, exist_ok=True)
    
    print(out_pth_json)    

\\kittyserverdw\Andre_kit\data\BTC\H1\StarDist_8_28_2025_pdac\json


In [3]:
out_pth_analysis_stats = os.path.join(out_pth_analysis, 'stats')
out_pth_analysis_boxplots = os.path.join(out_pth_analysis, 'boxplots')
out_pth_analysis_histograms = os.path.join(out_pth_analysis, 'histograms')
out_pth_analysis_scatterplots = os.path.join(out_pth_analysis, 'scatterplots')
out_pth_analysis_violin = os.path.join(out_pth_analysis, 'violinplots')
out_pth_analysis_densitycurves = os.path.join(out_pth_analysis, 'density_curves')
out_pth_analysis_heatmap = os.path.join(out_pth_analysis, 'heatmap')
    
# Create directories if they don't exist
for path in [out_pth_analysis_stats,out_pth_analysis_boxplots,out_pth_analysis_histograms,
             out_pth_analysis_scatterplots,out_pth_analysis_violin,out_pth_analysis_densitycurves,out_pth_analysis_heatmap]:
    os.makedirs(path, exist_ok=True)

In [4]:
out_pth_analysis_violin

'\\\\kittyserverdw\\Andre_kit\\data\\BTC\\H1\\StarDist_8_28_2025_pdac\\analysis\\violinplots'

In [5]:
# path 10x H&E stained images
# pth10x=r'\\169.254.138.20\Andre\data\Stardist\Healthy nPOD\nondiabetic\10x'
# pth10x=r'\\kittyserverdw\Andre_kit\data\BTC\H15\10x_python'
pth10x=os.path.join(WSI_path,'10x_python')
print(pth10x)

# path to the 10x segmented images
# pth10xsegmented=os.path.join(pth10x, 'classification_12152024_HE')
pth10xsegmented=os.path.join(pth10x, 'classification_10_5_2023')
# pth10xsegmented=os.path.join(pth10x, 'classification_01_25_2024_10x_PNET')

# Class names and colors
# class_names = ["islets of Langerhans", "pancreatic ducts", "blood vessels", "fat",
#                "acini", "stroma", "background", "nerves"]
# class_colors = [
#     [70, 220, 220], [80, 70, 220], [70, 180, 80], [220, 220, 80],
#     [130, 50, 190], [240, 180, 230], [255, 255, 255], [160, 130, 40]
# ]
class_names = ["tumor", "stroma", "immune", "vasculature", "fat", "white", "acini", "islets", "nerve", "duct"]
class_colors = [
    [253, 146, 87],    # 1 tumor (orange)
    [224, 187, 228],   # 2 stroma (light pink)
    [0, 30, 30],       # 3 immune (black)
    [234, 0, 0],       # 4 vasculature (red)
    [255, 255, 0],     # 5 fat (yellow)
    [255, 255, 255],   # 6 white (white)
    [149, 35, 184],    # 7 acini (purple)
    [247, 243, 203],   # 8 islets (light beige)
    [205, 235, 204],   # 9 nerve (light green)
    [0, 0, 255]        # 10 duct (blue)
]
# class_names = ["islets_of_Langerhans", "normal_epithelium", "smooth_muscle", "fat", "acini", "stroma", "nontissue", "PanIN", "nerves", "immune_hotspots", "endothelium", "PanNET"]
# class_colors=[
#      [70,220,220],  # 1 islet
#      [80,70,220],  # 2 duct
#      [70,180,80],  # 3 smooth muscle
#      [220,220,80],  # 4 fat
#      [130,50,190],  # % 5 acinus
#      [240,180,230],  # 6 connective tissue
#      [255,255,255],  # 7 whitespace
#      [210,80,75 ],  # 8 PanIN
#      [160,130,40],  # 9 nerves
#      [50,50,50],  # 10 immune
#      [160,30,120],  # 11 endothelium
#      [15,80,110]   # 12 PanNET
# ]
class_colors_norm = [[c / 255 for c in color] for color in class_colors]

# pixel resolution of the segmented images from first matfile .mat in output_pixres
pthmatfiles = get_sorted_files(output_pixres, '.mat')
mat = loadmat(pthmatfiles[0])
res = mat['pix_res'][0]  # Extract the 'pix_res' array

# Extract resolution values from the nested structure
pixelres = float(res[0][0][0])

pthjsonfiles = get_sorted_files(out_pth_json, '.json')
WSIs = get_sorted_files(WSI_path, '.svs','.ndpi')
dfs = get_sorted_files(outpthpkl, '.pkl')

\\kittyserverdw\Andre_kit\data\BTC\H1\10x_python


In [6]:
# make geojson contours of the segmented cells with respective classes
make_geojson_contours_with_classes(
    out_pth_json= out_pth_json,
    ds_amt=1,
    ds_mask=1/pixelres, # 1/0.4416->10x usually;
    class_names=class_names,
    class_colors=class_colors_norm,
    gj=1,
    pth10xsegmented=pth10xsegmented,
    step=50
)

1 / 173
H15_0001.json
Skipping \\kittyserverdw\Andre_kit\data\BTC\H15\StarDist_8_28_2025_pdac\json\geojsons\32_polys_20x_labels\H15_0001.geojson
51 / 173
H15_0151.json
Skipping \\kittyserverdw\Andre_kit\data\BTC\H15\StarDist_8_28_2025_pdac\json\geojsons\32_polys_20x_labels\H15_0151.geojson
101 / 173
H15_0301.json
Skipping \\kittyserverdw\Andre_kit\data\BTC\H15\StarDist_8_28_2025_pdac\json\geojsons\32_polys_20x_labels\H15_0301.geojson
151 / 173
H15_0451.json
Skipping \\kittyserverdw\Andre_kit\data\BTC\H15\StarDist_8_28_2025_pdac\json\geojsons\32_polys_20x_labels\H15_0451.geojson


In [ ]:
# EXTRACTS FEATURES FROM JSON CONTOURS AND SAVES THEM IN A PICKLE DATAFRAME
# computes also the RGB average intensities inside of each contour 
make_RGBfeatures_df_pkl_from_contours_jsons_with_classes(pthjsonfiles, WSIs, outpthpkl, segmented_image_dir=pth10xsegmented, ds_mask=1/pixelres,class_names=class_names)

In [ ]:
# MAKE LIST OF FEATURES DF PKL FILES AND SAVES THEM AS MAT FILES 
featuresdf_pkl2mat(outpthpkl, outpthmat, dfs)

## Load pkl dataframe with all cell features

In [6]:
sns.set(style="ticks")
# Convert RGB colors to HEX
class_colors_hex = rgb_to_hex(class_colors)
# Create a palette dictionary mapping class names to HEX colors
palette = dict(zip(class_names, class_colors_hex))
# Class Mapping: Integer labels to Class Names
class_mapping = {
    1: "islets of Langerhans",
    2: "pancreatic ducts",
    3: "blood vessels",
    4: "fat",
    5: "acini",
    6: "stroma",
    7: "background",
    8: "nerves"
}

In [7]:
# outpthpkl = os.path.join(out_pth, 'stardist_feature_df_pickles_classes')
outpthpkl = os.path.join(out_pth, 'stardist_feature_df_pickles')
# Load and combine data
df = load_data(outpthpkl)

Loading 100 .pkl files: 100%|██████████| 100/100 [00:25<00:00,  3.98file/s]


Combined DataFrame shape: (18517389, 23)


In [10]:
print(outpthpkl)

\\kittyserverdw\Andre_kit\data\BTC\H1\StarDist_8_28_2025_pdac\stardist_feature_df_pickles


In [8]:
np.shape(df)

(18517389, 23)

In [9]:
df.head()

,Centroid_x,Centroid_y,Area,Perimeter,Circularity,Aspect Ratio,compactness,eccentricity,extent,form_factor,...,minor_axis_length,major_axis_length,orientation_degrees,r_mean_intensity,g_mean_intensity,b_mean_intensity,r_std,g_std,b_std,slide_num
0,10410.0,3340.0,79.089012,34.601498,0.830111,1.537583,15.138179,0.759616,0.765381,1.204658,...,8.197850,12.604875,9864.273438,144.559998,130.779999,167.009995,23.908043,21.856510,27.213070,1.0
1,10516.0,3048.0,94.448509,38.174858,0.814423,1.759907,15.429779,0.822882,0.763537,1.227863,...,8.383737,14.754597,7721.419434,180.470001,179.500000,182.479996,32.539051,32.238895,32.715919,1.0
2,10496.0,3034.0,144.009613,47.940079,0.787415,2.137907,15.959011,0.883862,0.768479,1.269978,...,9.362362,20.015856,6247.031250,185.630005,183.050003,187.639999,39.885487,39.198547,40.205673,1.0
3,13320.0,22.0,47.217121,26.739437,0.829860,1.407134,15.142758,0.703531,0.765649,1.205022,...,6.620140,9.315425,1799.306885,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,1.0
4,13328.0,26.0,42.541756,25.594856,0.816056,1.175133,15.398911,0.525218,0.748528,1.225406,...,6.954410,8.172356,462.186768,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,1.0


In [11]:
# Define numerical features to analyze (exclude 'slide_num' and 'class_name')
numerical_features = [col for col in df.columns if col not in ['slide_num', 'class_name', 'Centroid_x', 'Centroid_y']]

In [ ]:
import os
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

def calculate_statistics(df, feature, class_mapping):
    """
    Calculate statistics (mean, median, std, etc.) for a given feature across classes.

    Parameters:
        df (pd.DataFrame): DataFrame containing the data with 'class_name' and the feature.
        feature (str): The feature to calculate statistics for.
        class_mapping (dict): Mapping from class integers to class names.

    Returns:
        pd.DataFrame: A DataFrame containing statistics for each class.
    """
    # Exclude class 7 ("background")
    filtered_df = df[df['class_name'] != 7].copy()

    # Map class integers to class names
    filtered_df['class_name_str'] = filtered_df['class_name'].map(class_mapping)

    # Drop any rows where mapping resulted in NaN (i.e., classes not in class_mapping)
    filtered_df = filtered_df.dropna(subset=['class_name_str'])

    # Group by class and calculate statistics
    stats_df = filtered_df.groupby('class_name_str')[feature].agg(
        mean=np.mean,
        median=np.median,
        std=np.std,
        min=np.min,
        max=np.max,
        count='count'
    ).reset_index()

    return stats_df

def plot_violin_shifted(df, feature, class_mapping, palette, output_dir):
    """
    Create a violin plot for a given feature across classes, excluding class 7,
    sorted by median descending. Plot data points shifted to the right of violins.

    Parameters:
        df (pd.DataFrame): DataFrame containing the data with 'class_name' and the feature.
        feature (str): The feature to plot.
        class_mapping (dict): Mapping from class integers to class names.
        palette (dict): Mapping from class names to colors.
        output_dir (str): Path to save the plot and statistics CSV.
    """
    # Define the resolution for pixel-to-micron conversion (0.504 microns/pixel)
    resolution = 0.504  # microns per pixel

    # Ensure the output directory exists
    os.makedirs(output_dir, exist_ok=True)

    # 1. Exclude class 7 ("background")
    filtered_df = df[df['class_name'] != 7].copy()

    if filtered_df.empty:
        print("No data available after excluding Class 7 ('background').")
        return

    # 2. Map class integers to class names
    filtered_df['class_name_str'] = filtered_df['class_name'].map(class_mapping)

    # Drop any rows where mapping resulted in NaN (i.e., classes not in class_mapping)
    filtered_df = filtered_df.dropna(subset=['class_name_str'])

    # 3. Convert pixel-based metrics to microns
    if feature in ['Area', 'Perimeter', 'maximum_radius', 'mean_radius', 'median_radius',
                   'minor_axis_length', 'major_axis_length']:
        # Apply pixel-to-micron conversion
        if feature == 'Area':
            filtered_df[feature] = filtered_df[feature] * (resolution ** 2)  # Area scales with resolution squared
        else:
            filtered_df[feature] = filtered_df[feature] * resolution  # Linear metrics scale with resolution

    # 4. Calculate median per class and sort classes descending
    median_per_class = filtered_df.groupby('class_name_str')[feature].median().sort_values(ascending=False)
    sorted_classes = median_per_class.index.tolist()

    # 5. Set the order for the plot
    sns.set(style="ticks")
    plt.figure(figsize=(12, 8))

    # 6. Create violin plot
    ax = sns.violinplot(
        x='class_name_str',
        y=feature,
        data=filtered_df,
        order=sorted_classes,
        palette=palette,
        inner='quartile',
        width=0.35,
        linewidth=2.5
    )

    # 7. Shift data points to the right
    shift = 0.5  # Amount to shift data points to the right
    for i, class_name in enumerate(sorted_classes):
        subset = filtered_df[filtered_df['class_name_str'] == class_name]
        # Generate jittered x positions shifted to the right
        x = np.random.normal(loc=i, scale=0.05, size=len(subset)) + shift
        scatter_color = palette.get(class_name, 'black')
        plt.scatter(x, subset[feature], color=scatter_color, s=0.5, alpha=0.3)

    # 8. Apply despine for a clean look
    sns.despine()

    # 9. Rotate x labels for better readability
    plt.xticks(rotation=45, fontsize=18)
    plt.yticks(fontsize=18)

    # 10. Set titles and labels
    plt.title(f'Violin Plot of {feature} by Class', fontsize=20)
    plt.xlabel('Class Name', fontsize=20)
    plt.ylabel(f'{feature} ({ "µm²" if feature == "Area" else "µm" })', fontsize=20)  # Add units

    # 11. Adjust layout for better spacing
    plt.tight_layout()

    # 12. Save the plot as PNG
    plot_path = os.path.join(output_dir, f'violin_{feature}.png')
    plt.savefig(plot_path, dpi=300)
    plt.close()
    print(f"Violin plot saved to {plot_path}")

In [ ]:
df.columns

In [ ]:
numerical_features

In [ ]:
# Define the directory containing the CSV files
csv_directory = out_pth_analysis_violin # Update this path to your directory

# Define the features to plot
sns.set(style="ticks")
# plt.style.use("custom.mplstyle")
resolution = 0.504  # microns per pixel

# Create a figure for plotting
plt.figure(figsize=(15, 5))

# Ensure the output directory for SVG files exists
output_svg_dir = os.path.join(csv_directory, 'svg_plots')
os.makedirs(output_svg_dir, exist_ok=True)

# Loop through each feature and plot the mean ± std
for feature in numerical_features:
    # Load the CSV file
    csv_path = os.path.join(csv_directory, f'stats_{feature}.csv')
    if not os.path.exists(csv_path):
        print(f"CSV file for {feature} not found at {csv_path}. Skipping.")
        continue

    # Read the CSV file
    stats_df = pd.read_csv(csv_path)

    # Convert pixel-based metrics to microns
    if feature in ['Area', 'Perimeter', 'maximum_radius', 'mean_radius', 'median_radius',
                   'minor_axis_length', 'major_axis_length']:
        # Convert mean and std to microns
        stats_df['mean'] = stats_df['mean'] * (resolution ** 2 if feature == 'Area' else resolution)
        stats_df['std'] = stats_df['std'] * (resolution ** 2 if feature == 'Area' else resolution)

    # Sort the DataFrame by mean value (highest to lowest)
    stats_df = stats_df.sort_values(by='mean', ascending=False)

    # Create a new figure for each feature
    plt.figure(figsize=(10, 6))

    # Plot the mean values with error bars (std) and use colors from the palette
    for _, row in stats_df.iterrows():
        class_name = row['class_name_str']
        plt.errorbar(
            class_name,  # x-axis: class name
            row['mean'],  # y-axis: mean value
            yerr=row['std'],  # error bars: standard deviation
            fmt='o',  # marker style
            capsize=18,  # error bar cap size
            linestyle='none',  # no connecting lines
            markersize=18,  # marker size
            color=palette.get(class_name, 'black'),  # use color from palette
            label=f'{class_name}'
        )

    # Add labels and title
    plt.xlabel('Class Name', fontsize=18)
    plt.ylabel(f'{feature} ({ "µm²" if feature == "Area" else "µm" })', fontsize=18)
    plt.title(f'Mean {feature} with Standard Deviation', fontsize=14)
    plt.xticks(rotation=60, fontsize=18)  # Rotate x-axis labels for readability

    # Remove top and right spines (box off)
    sns.despine()

    # Make the bottom and left axis lines thicker
    for spine in plt.gca().spines.values():
        spine.set_linewidth(2)  # Set the linewidth to 2 (or any desired thickness)

    # Make the ticks thicker (same thickness as axis lines)
    plt.tick_params(axis='both', which='both', width=2)  # Set tick thickness to 2

    # Make the y-axis tick labels larger
    plt.tick_params(axis='y', labelsize=18)  # Set y-axis tick label font size to 14

    # Add a legend
    plt.legend(fontsize=18, bbox_to_anchor=(1.05, 1), loc='upper left')

    # Adjust layout
    plt.tight_layout()

    # Save the plot as an SVG file
    svg_path = os.path.join(output_svg_dir, f'mean_{feature}_with_std.svg')
    plt.savefig(svg_path, format='svg', dpi=500)
    plt.close()
    print(f"Plot saved to {svg_path}")

##NEW CODES


In [ ]:
import os
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

def calculate_statistics(df, feature, class_mapping):
    """
    Calculate statistics (mean, median, std, etc.) for a given feature across classes.
    Also computes the coefficient of variation (CV = std/mean).

    Parameters:
        df (pd.DataFrame): DataFrame containing the data with 'class_name' and the feature.
        feature (str): The feature to calculate statistics for.
        class_mapping (dict): Mapping from class integers to class names.

    Returns:
        pd.DataFrame: A DataFrame containing statistics for each class.
    """
    # Exclude class 7 ("background") and map class names
    filtered_df = df[df['class_name'] != 7].copy()
    filtered_df['class_name_str'] = filtered_df['class_name'].map(class_mapping)
    filtered_df = filtered_df.dropna(subset=['class_name_str'])

    # Group by class and calculate statistics
    stats_df = filtered_df.groupby('class_name_str')[feature].agg(
        mean=np.mean,
        median=np.median,
        std=np.std,
        min=np.min,
        max=np.max,
        count='count'
    ).reset_index()

    # Calculate coefficient of variation (CV = std/mean), avoiding division by zero
    stats_df['coef_var'] = stats_df.apply(lambda row: row['std'] / row['mean'] if row['mean'] != 0 else np.nan, axis=1)

    return stats_df

def plot_violin_shifted(df, feature, class_mapping, palette, output_dir):
    """
    Create a violin plot for a given feature across classes (excluding class 7) sorted by median descending.
    Also computes and saves statistics—including coefficient of variation—to CSV and Excel files.

    Parameters:
        df (pd.DataFrame): DataFrame with 'class_name' and the feature.
        feature (str): The feature to plot.
        class_mapping (dict): Mapping from class integers to class names.
        palette (dict): Mapping from class names to colors.
        output_dir (str): Directory to save the plot and statistics files.
    """
    resolution = 0.504  # microns per pixel
    os.makedirs(output_dir, exist_ok=True)

    # Filter and map class names
    filtered_df = df[df['class_name'] != 7].copy()
    filtered_df['class_name_str'] = filtered_df['class_name'].map(class_mapping)
    filtered_df = filtered_df.dropna(subset=['class_name_str'])

    # Convert pixel-based metrics to microns if applicable
    pixel_features = ['Area', 'Perimeter', 'maximum_radius', 'mean_radius',
                      'median_radius', 'minor_axis_length', 'major_axis_length']
    if feature in pixel_features:
        if feature == 'Area':
            filtered_df[feature] = filtered_df[feature] * (resolution ** 2)
        else:
            filtered_df[feature] = filtered_df[feature] * resolution

    # Calculate statistics and convert them if needed
    stats_df = calculate_statistics(df, feature, class_mapping)
    if feature in pixel_features:
        factor = resolution ** 2 if feature == 'Area' else resolution
        stats_df['mean']   = stats_df['mean'] * factor
        stats_df['std']    = stats_df['std'] * factor
        stats_df['median'] = stats_df['median'] * factor
        stats_df['min']    = stats_df['min'] * factor
        stats_df['max']    = stats_df['max'] * factor
        # Note: coef_var remains the same because it is a ratio

    # Save statistics to CSV and Excel (CV is included)
    stats_csv_path = os.path.join(output_dir, f'stats_{feature}.csv')
    stats_df.to_csv(stats_csv_path, index=False)
    print(f"Statistics saved to {stats_csv_path}")

    stats_excel_path = os.path.join(output_dir, f'stats_{feature}.xlsx')
    stats_df.to_excel(stats_excel_path, index=False)
    print(f"Statistics saved to {stats_excel_path}")

    # Sort classes by median descending for plotting
    median_per_class = filtered_df.groupby('class_name_str')[feature].median().sort_values(ascending=False)
    sorted_classes = median_per_class.index.tolist()

    sns.set(style="ticks")
    plt.figure(figsize=(12, 8))

    # Create violin plot
    ax = sns.violinplot(
        x='class_name_str',
        y=feature,
        data=filtered_df,
        order=sorted_classes,
        palette=palette,
        inner='quartile',
        width=0.35,
        linewidth=2.5
    )

    # Plot shifted data points
    shift = 0.5
    for i, class_name in enumerate(sorted_classes):
        subset = filtered_df[filtered_df['class_name_str'] == class_name]
        x = np.random.normal(loc=i, scale=0.05, size=len(subset)) + shift
        scatter_color = palette.get(class_name, 'black')
        plt.scatter(x, subset[feature], color=scatter_color, s=0.5, alpha=0.3)

    sns.despine()
    plt.xticks(rotation=45, fontsize=18)
    plt.yticks(fontsize=18)
    plt.title(f'Violin Plot of {feature} by Class', fontsize=20)
    plt.xlabel('Class Name', fontsize=20)
    plt.ylabel(f'{feature} ({"µm²" if feature=="Area" else "µm"})', fontsize=20)
    plt.tight_layout()

    plot_path = os.path.join(output_dir, f'violin_{feature}.png')
    plt.savefig(plot_path, dpi=300)
    plt.close()
    print(f"Violin plot saved to {plot_path}")



In [ ]:
def plot_violin_shifted(df, feature, class_mapping, palette, output_dir, plot=1):
    """
    Create a violin plot for a given feature across classes (excluding class 7) sorted by median descending.
    Also computes and saves statistics—including coefficient of variation—to CSV and Excel files.

    Parameters:
        df (pd.DataFrame): DataFrame with 'class_name' and the feature.
        feature (str): The feature to plot.
        class_mapping (dict): Mapping from class integers to class names.
        palette (dict): Mapping from class names to colors.
        output_dir (str): Directory to save the plot and statistics files.
        plot (int): 1 to plot and save the violin plot; 0 to skip plotting.
    """
    resolution = 0.504  # microns per pixel
    os.makedirs(output_dir, exist_ok=True)

    # Filter and map class names
    filtered_df = df[df['class_name'] != 7].copy()
    filtered_df['class_name_str'] = filtered_df['class_name'].map(class_mapping)
    filtered_df = filtered_df.dropna(subset=['class_name_str'])

    # Convert pixel-based metrics to microns if applicable
    pixel_features = ['Area', 'Perimeter', 'maximum_radius', 'mean_radius',
                      'median_radius', 'minor_axis_length', 'major_axis_length']
    if feature in pixel_features:
        if feature == 'Area':
            filtered_df[feature] = filtered_df[feature] * (resolution ** 2)
        else:
            filtered_df[feature] = filtered_df[feature] * resolution

    # Calculate statistics and convert them if needed
    stats_df = calculate_statistics(df, feature, class_mapping)
    if feature in pixel_features:
        factor = resolution ** 2 if feature == 'Area' else resolution
        stats_df['mean']   = stats_df['mean'] * factor
        stats_df['std']    = stats_df['std'] * factor
        stats_df['median'] = stats_df['median'] * factor
        stats_df['min']    = stats_df['min'] * factor
        stats_df['max']    = stats_df['max'] * factor
        # Note: coef_var remains the same because it is a ratio

    # Save statistics to CSV and Excel (CV is included)
    stats_csv_path = os.path.join(output_dir, f'stats_{feature}.csv')
    stats_df.to_csv(stats_csv_path, index=False)
    print(f"Statistics saved to {stats_csv_path}")

    stats_excel_path = os.path.join(output_dir, f'stats_{feature}.xlsx')
    stats_df.to_excel(stats_excel_path, index=False)
    print(f"Statistics saved to {stats_excel_path}")

    # If plotting is requested, create and save the violin plot
    if plot:
        # Sort classes by median descending for plotting
        median_per_class = filtered_df.groupby('class_name_str')[feature].median().sort_values(ascending=False)
        sorted_classes = median_per_class.index.tolist()

        sns.set(style="ticks")
        plt.figure(figsize=(12, 8))

        # Create violin plot
        ax = sns.violinplot(
            x='class_name_str',
            y=feature,
            data=filtered_df,
            order=sorted_classes,
            palette=palette,
            inner='quartile',
            width=0.35,
            linewidth=2.5
        )

        # Plot shifted data points
        shift = 0.5
        for i, class_name in enumerate(sorted_classes):
            subset = filtered_df[filtered_df['class_name_str'] == class_name]
            x = np.random.normal(loc=i, scale=0.05, size=len(subset)) + shift
            scatter_color = palette.get(class_name, 'black')
            plt.scatter(x, subset[feature], color=scatter_color, s=0.5, alpha=0.3)

        sns.despine()
        plt.xticks(rotation=45, fontsize=18)
        plt.yticks(fontsize=18)
        plt.title(f'Violin Plot of {feature} by Class', fontsize=20)
        plt.xlabel('Class Name', fontsize=20)
        plt.ylabel(f'{feature} ({"µm²" if feature=="Area" else "µm"})', fontsize=20)
        plt.tight_layout()

        plot_path = os.path.join(output_dir, f'violin_{feature}.png')
        plt.savefig(plot_path, dpi=300)
        plt.close()
        print(f"Violin plot saved to {plot_path}")
    else:
        print("Plotting skipped..")


In [ ]:
# plot_violin_shifted(df, numerical_features, class_mapping, palette, out_pth_analysis_violin)
# Iterate over numerical features and plot
for feature in numerical_features:
    # plot_violin_shifted(df, feature, class_mapping, palette, out_pth_analysis_violin,plot=1)
    plot_violin_shifted(df, feature, class_mapping, palette, out_pth_analysis_violin,plot=0)

In [ ]:

# -------------------------------
# Plotting mean ± std and coefficient of variation from saved statistics
# -------------------------------

# Define the directory containing the CSV/Excel files (update this path accordingly)
csv_directory = out_pth_analysis_violin  # Update this path to your directory

# Define numerical features and the palette (update as needed)
numerical_features = ['Area', 'Perimeter', 'maximum_radius', 'mean_radius',
                      'median_radius', 'minor_axis_length', 'major_axis_length']

# Ensure the output directory for SVG plots exists
output_svg_dir = os.path.join(csv_directory, 'svg_plots')
os.makedirs(output_svg_dir, exist_ok=True)

for feature in numerical_features:
    csv_path = os.path.join(csv_directory, f'stats_{feature}.csv')
    if not os.path.exists(csv_path):
        print(f"CSV file for {feature} not found at {csv_path}. Skipping.")
        continue

    # Read the saved statistics (which include the coefficient of variation)
    stats_df = pd.read_csv(csv_path)

    # Plot Mean ± Standard Deviation
    plt.figure(figsize=(10, 6))
    for _, row in stats_df.iterrows():
        class_name = row['class_name_str']
        plt.errorbar(
            class_name,            # x-axis: class name
            row['mean'],           # y-axis: mean value
            yerr=row['std'],       # error bars: standard deviation
            fmt='o',               # marker style
            capsize=18,            # error bar cap size
            linestyle='none',      # no connecting lines
            markersize=18,         # marker size
            color=palette.get(class_name, 'black'),
            label=f'{class_name}'
        )
    plt.xlabel('Class Name', fontsize=18)
    plt.ylabel(f'{feature} ({"µm²" if feature=="Area" else "µm"})', fontsize=18)
    plt.title(f'Mean {feature} with Standard Deviation', fontsize=14)
    plt.xticks(rotation=60, fontsize=18)
    sns.despine()
    for spine in plt.gca().spines.values():
        spine.set_linewidth(2)
    plt.tick_params(axis='both', which='both', width=2)
    plt.tick_params(axis='y', labelsize=18)
    plt.legend(fontsize=18, bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.tight_layout()
    svg_path = os.path.join(output_svg_dir, f'mean_{feature}_with_std.svg')
    plt.savefig(svg_path, format='svg', dpi=500)
    plt.close()
    print(f"Mean ± Std plot saved to {svg_path}")

    # Plot Coefficient of Variation (CV)
    plt.figure(figsize=(10, 6))
    for _, row in stats_df.iterrows():
        class_name = row['class_name_str']
        plt.scatter(
            class_name,
            row['coef_var'],
            color=palette.get(class_name, 'black'),
            s=100,
            label=f'{class_name}'
        )
    plt.xlabel('Class Name', fontsize=18)
    plt.ylabel('Coefficient of Variation', fontsize=18)
    plt.title(f'Coefficient of Variation of {feature}', fontsize=14)
    plt.xticks(rotation=60, fontsize=18)
    sns.despine()
    plt.legend(fontsize=18, bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.tight_layout()
    cv_svg_path = os.path.join(output_svg_dir, f'cv_{feature}.svg')
    plt.savefig(cv_svg_path, format='svg', dpi=500)
    plt.close()
    print(f"Coefficient of Variation plot saved to {cv_svg_path}")
